# SEM 畸变校正 — Jupyter 交互版

本 notebook 与命令行工具**完全等价**：底层都调用同一个 `semcorr.correct_image()` 库 API。

| 命令行 | 本 notebook |
|---|---|
| `./semcorr image.tif --grid 2x3` | 第 1–6 节 |
| `./semcorr --batch image_folder` | 第 7 节 |

相比 CLI 的额外好处：输入图像预览、四张诊断图**内联显示**、报告摘要表格、批量处理汇总表，全部在一个页面里完成，不用再手动打开 PNG。

**依赖**：`numpy`、`opencv-python-headless`、`matplotlib`（与 CLI 相同）。
真实 SEM 图像不随仓库分发，因此默认使用与回归测试 `tests/test_regression.py` 同款的**合成演示图**（已知 3° 旋转 × 各向异性缩放 + 透视畸变）；把 `IMAGE_PATH` 指向真实图像即可无缝切换。

In [ ]:
import sys
import importlib.util
from pathlib import Path

# 自动定位失败时，把这里改为项目绝对路径，例如：
# PROJECT_ROOT = Path("/Users/asphyxia/Research/Kitave/sem-map-corrector")
PROJECT_ROOT = None

def _find_project_root():
    """从 notebook 位置 / 内核工作目录出发，向上逐级搜索 src/semcorr。"""
    starts = []
    if "__vsc_ipynb_file__" in globals():   # VS Code/TRAE 注入的 notebook 路径
        starts.append(Path(globals()["__vsc_ipynb_file__"]).parent)
    try:                                    # IPython 目录历史（内核工作目录）
        starts.extend(Path(d) for d in _dh)
    except NameError:
        pass
    starts.append(Path.cwd())
    for start in starts:
        for cand in (start, *start.parents):        # 逐级向上找
            if (cand / "src" / "semcorr").is_dir():
                return cand
            sub = cand / "sem-map-corrector"        # 工作区根在项目上一级时
            if (sub / "src" / "semcorr").is_dir():
                return sub
    return None

ROOT = PROJECT_ROOT or _find_project_root()
if ROOT is not None:
    sys.path.insert(0, str(ROOT / "src"))           # 直接用源码，无需安装

try:
    from semcorr import correct_image   # 包入口是懒加载，此处不触发重依赖
except ModuleNotFoundError as exc:
    raise RuntimeError(
        "找不到 semcorr 包。解决办法任选其一：\n"
        "  1. 确认本 notebook 位于 sem-map-corrector/ 内，或在其上层目录打开的工作区中\n"
        "  2. 把上方 PROJECT_ROOT 改为项目绝对路径后重跑本 cell\n"
        "  3. 在项目根目录执行 `python -m pip install -e .` 后重跑"
    ) from exc

# 依赖自检：缺包时给出可操作指引（而不是裸 ModuleNotFoundError）
_missing = [pip for mod, pip in
            (("cv2", "opencv-python-headless"), ("numpy", "numpy"),
             ("matplotlib", "matplotlib"))
            if importlib.util.find_spec(mod) is None]
if _missing:
    raise RuntimeError(
        "当前内核缺少依赖: " + ", ".join(_missing) + "\n"
        "内核 Python: " + sys.executable + "\n"
        "解决办法任选其一：\n"
        "  1. 在 IDE 右上角内核选择器切换到已装齐依赖的 Python（推荐 miniconda3），"
        "重启内核并运行全部\n"
        "  2. 在新 cell 执行安装（装进当前内核的环境）：\n"
        "     !{\"" + sys.executable + "\"} -m pip install " + " ".join(_missing)
    )
del _missing

# 恢复 notebook 内联绘图（semcorr 导入时会把 matplotlib 后端切到 Agg）
%matplotlib inline

import cv2
import numpy as np
import matplotlib.pyplot as plt

print("semcorr 导入成功，项目根目录:", ROOT)
print("内核 Python:", sys.executable)


## 1. 配置参数

| 参数 | 含义 |
|---|---|
| `IMAGE_PATH` | 输入图像；`None` = 自动生成合成演示图 |
| `GRID` | 标记网格 行x列（标准版图 `2x2`；演示图为 `2x3`） |
| `OUTDIR` | 输出目录：校正图 + `diagnostics/` |
| `DESIGN` | 设计坐标 JSON（可选；`None` = 由检测点推断正方形网格） |
| `AFFINE` | `False` = 分格精确单应（默认，mark 中心严格正方形）；`True` = 全局仿射 |

In [ ]:
# ===== 在这里修改参数 =====
IMAGE_PATH = None                  # None = 用合成演示图；正式使用改为 Path("/path/to/image.tif")
GRID = "2x2"                       # 标记网格 行x列（标准 4-mark 版图；6-mark 用 2x3）
OUTDIR = ROOT / "notebook_output"  # 输出目录
DESIGN = None                      # 设计坐标 JSON 路径（可选）
AFFINE = False                     # True = 全局仿射；False = 分格精确单应（默认）

n_rows, n_cols = (int(v) for v in GRID.lower().split("x"))
print(f"网格: {n_rows} 行 x {n_cols} 列，共 {n_rows * n_cols} 个标记")

## 2. 演示数据（`IMAGE_PATH = None` 时自动执行）

真实实验图像不随仓库分发。这里程序化渲染一张合成 SE2 图像：正方形十字网格 + 照明梯度 + 已知畸变（3° 旋转、各向异性缩放 1.02/0.98、平移、弱透视），用于无实验数据时验证全流程。正式使用时设好 `IMAGE_PATH` 后本节自动跳过。

In [ ]:
import math

def make_demo_image(path, n_rows=2, n_cols=3, img_w=640, img_h=420,
                    margin=70.0, pitch=160.0, span=60, arm=12):
    """渲染带照明梯度 + 已知几何畸变的合成 SE2 十字网格图。"""
    gradient = 45 + 25.0 * np.arange(img_w) / img_w        # 水平照明梯度
    img = np.repeat(gradient.astype(np.uint8)[None, :], img_h, axis=0)
    ideal = [(margin + c * pitch, margin + r * pitch)
             for r in range(n_rows) for c in range(n_cols)]
    for x, y in ideal:                                      # 画实心亮十字
        xi, yi = int(round(x)), int(round(y))
        h = arm // 2
        x0, y0 = xi - span // 2, yi - span // 2
        img[yi - h:yi + h + 1, x0:x0 + span] = 210
        img[y0:y0 + span, xi - h:xi + h + 1] = 210
    th = math.radians(3.0)                                  # 已知畸变
    A = (np.array([[math.cos(th), -math.sin(th)],
                   [math.sin(th), math.cos(th)]]) @ np.diag([1.02, 0.98]))
    H = np.eye(3)
    H[:2, :2] = A
    H[:2, 2] = [6.0, -4.0]
    H[2, 0], H[2, 1] = 2.0e-6, 1.5e-6
    distorted = cv2.warpPerspective(img, H, (img_w, img_h))
    if not cv2.imwrite(str(path), distorted):
        raise RuntimeError(f"演示图写入失败: {path}")
    return Path(path)

if IMAGE_PATH is None:
    OUTDIR.mkdir(parents=True, exist_ok=True)
    IMAGE_PATH = make_demo_image(OUTDIR / f"demo_se2_{n_rows}x{n_cols}.tif",
                                 n_rows, n_cols)
    print("已生成演示图:", IMAGE_PATH)
else:
    IMAGE_PATH = Path(IMAGE_PATH)
    print("使用指定图像:", IMAGE_PATH)

## 3. 输入预览

确认图像类型为 SE2 实心亮十字（本项目不做自动风格切换，InLens 浮雕风格不在支持范围内）。

In [ ]:
gray = cv2.imread(str(IMAGE_PATH), cv2.IMREAD_GRAYSCALE)
if gray is None:
    raise FileNotFoundError(f"无法读取图像: {IMAGE_PATH}")

fig, ax = plt.subplots(figsize=(9, 6))
ax.imshow(gray, cmap="gray")
ax.set_title(f"输入: {IMAGE_PATH.name}  ({gray.shape[1]} x {gray.shape[0]})")
ax.axis("off")
plt.show()

## 4. 运行校正

等价于 `./semcorr <IMAGE_PATH> --grid <GRID>`。日志依次给出：粗筛候选 → 模板/对称/形状验证 → 网格指派 → 理想坐标 → 仿射/单应拟合与分解 → 留一交叉验证 → （必要时）网格自愈 → 校正后自检。

失败会直接抛 `RuntimeError`（标记数不足、拟合 RMS 超限、指派不合理等），错误信息里带修复建议。

In [ ]:
report = correct_image(
    IMAGE_PATH,
    grid=GRID,
    design=str(DESIGN) if DESIGN is not None else None,
    outdir=str(OUTDIR),
    affine=AFFINE,
)

## 5. 结果内联查看

四张输出图直接显示在下方（文件同时保存在 `OUTDIR` 与 `OUTDIR/diagnostics/`）：

- **校正图**：畸变校正后的图像，mark 中心严格构成正方形网格（默认分格精确单应模型）
- **检测诊断**：绿色圈 = 接受的候选中心，红色 × = 被拒绝的干扰物
- **残差诊断**：左图箭头为各 mark 残差方向（放大 20 倍），橙色圈 = 留一验证可疑点；右图为逐 mark 残差柱状图
- **中心标注图**：最终定位坐标标注（定位实验直接引用 `diagnostics/*_centers.csv`）

In [ ]:
outs = report["outputs"]

def _load(path):
    img = cv2.imread(str(path), cv2.IMREAD_UNCHANGED)
    if img is None:
        raise FileNotFoundError(f"无法读取: {path}")
    if img.ndim == 3 and img.shape[2] == 3:
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    return img

panels = [
    (outs["corrected_image"],   "校正图"),
    (outs["detection_overlay"], "检测诊断（绿=接受，红=拒绝）"),
    (outs["residual_plot"],     "残差诊断（箭头放大 20x，橙圈=留一可疑）"),
    (outs["centers_annotated"], "中心标注图"),
]
fig, axes = plt.subplots(2, 2, figsize=(18, 13))
for ax, (path, title) in zip(axes.ravel(), panels):
    ax.imshow(_load(path), cmap="gray")
    ax.set_title(title, fontsize=13)
    ax.axis("off")
plt.tight_layout()
plt.show()

## 6. 报告摘要

完整机器可读报告在 `OUTDIR/diagnostics/<图像名>_report.json`，含仿射/单应参数、自愈日志、留一验证、自检明细、输入 SHA256 等。

In [ ]:
sc = report["self_check"]
print("质量状态     :", report["quality_status"],
      "" if not report["quality_warnings"] else report["quality_warnings"])
print("校正模型     :", report["method"])
print("网格         : %d 行 x %d 列" % tuple(report["grid"]))
print("理想坐标来源 :", report["ideal_source"])
print("拟合 RMS     : %.4f px（仿射 %.4f / 单应 %.4f）"
      % (report["used_model_rms_px"],
         report["affine"]["rms_px"], report["homography"]["rms_px"]))
d = report["affine"]["decomposed"]
print("仿射分解     : scale_x=%.5f scale_y=%.5f 旋转=%.3f° 正交偏差=%.4f°"
      % (d["scale_x"], d["scale_y"], d["rotation_deg"], d["non_orthogonal_deg"]))
print("校正后自检   : %d/%d 个标记重新检出，残差 RMS = %s"
      % (sc["n_detected"], sc["n_total"],
         "—" if sc["rms"] is None else "%.3f" % sc["rms"]))
if report["repair"]:
    print("网格自愈     : 逐出并回填 " + ", ".join(r["slot"] for r in report["repair"]))

print()
print("%-4s %12s %12s %12s %12s %10s" % ("编号", "检测x", "检测y", "理想x", "理想y", "残差(px)"))
for m in report["marks"]:
    r = (m["residual_px"][0] ** 2 + m["residual_px"][1] ** 2) ** 0.5
    tag = "   [网格恢复]" if m["recovered_from_grid_search"] else ""
    print("%-4s %12.3f %12.3f %12.3f %12.3f %10.3f%s"
          % (m["id"], m["detected_px"][0], m["detected_px"][1],
             m["ideal"][0], m["ideal"][1], r, tag))

## 7. 批量处理（可选）

等价于 `./semcorr --batch <BATCH_DIR>`。把 `BATCH_DIR` 设为图像文件夹即可；每张图的输出与单张模式相同，最后生成汇总表。

In [ ]:
BATCH_DIR = "/Users/asphyxia/Research/Kitave/Correction map/260907"   # 例如 Path("/path/to/SEM_images")；None = 跳过本节

if BATCH_DIR is not None:
    from semcorr.io import list_images
    files = list_images(BATCH_DIR)
    batch_out = Path(BATCH_DIR) / "corrected"
    print(f"批量处理 {len(files)} 张图像 → {batch_out}")
    summary = []
    for i, p in enumerate(files, 1):
        print(f"===== [{i}/{len(files)}] {p.name} =====")
        try:
            rep = correct_image(p, grid=GRID, outdir=str(batch_out), affine=AFFINE)
            summary.append((p.name, rep["quality_status"],
                            rep["used_model_rms_px"],
                            rep["self_check"]["n_detected"],
                            rep["self_check"]["n_total"]))
        except (RuntimeError, FileNotFoundError, ValueError) as exc:
            print("失败：", exc)
            summary.append((p.name, "FAIL", None, None, None))
    print()
    print("%-32s %-12s %10s %10s" % ("图像", "状态", "RMS(px)", "自检"))
    for name, status, rms, nd, nt in summary:
        print("%-32s %-12s %10s %10s"
              % (name[:32], status,
                 "—" if rms is None else "%.3f" % rms,
                 "—" if nd is None else "%d/%d" % (nd, nt)))
else:
    print("跳过批量处理（把 BATCH_DIR 设为图像文件夹路径即可启用）")

## 结果解读（重要，与 README 一致）

- JSON 中的四点单应残差为零只代表模型穿过四个输入点，**不单独证明中心正确**。
- 正式实验前应查看上方的检测诊断图与中心标注图，**确认绿色中心落在十字中心**；残差图用于核查离群点。
- 留一验证警告（橙色圈）：剔除后 RMS 显著下降的 mark 疑似离群，需人工核查。
- 校正后自检：在校正图上重新检测全部标记，残差应处于单点检测噪声量级；明显偏大说明校正有问题。
- 质量状态 `PASS` / `WARN_REVIEW`：后者表示有需要人工复核的警告（如部分标记未检出）。
- 输入图像**不会被修改**；所有输出写入 `OUTDIR`。